In [1]:
! pip install transformers accelerate torch datasets

In [14]:
import transformers
print(transformers.__version__)

5.0.0


In [4]:
import pandas as pd

df = pd.read_csv("/content/preprocessed_data.csv")
df = df[["cleaned_text", "label"]]
df = df.rename(columns={"cleaned_text":"text"})

In [5]:
df

,text,label
0,donald trump white house chaos trying cover ru...,0
1,donald trump presumptive gop nominee time reme...,0
2,mike penny huge homophobe support exgay conver...,0
3,san francisco reuters california attorney gene...,1
4,twisted reasoning come pelosi day especially p...,0
...,...,...
44183,abuja reuters united state formally agreed sel...,1
44184,tune alternate current radio network acr anoth...,0
44185,convinced freedom religion group atheist get e...,0
44186,washington reuters republican tax plan unveile...,1


In [6]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, stratify=df['label'], random_state=42)

In [7]:
from datasets import Dataset

train = Dataset.from_pandas(train_df.reset_index(drop=True))
test = Dataset.from_pandas(test_df.reset_index(drop=True))

In [8]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [9]:
def tokenize(batch):
  return tokenizer(batch['text'], padding="max_length", truncation=True, max_length=256)

In [10]:
train = train.map(tokenize, batched=True)
test = test.map(tokenize, batched=True)

Map:   0%|          | 0/35350 [00:00<?, ? examples/s]

Map:   0%|          | 0/8838 [00:00<?, ? examples/s]

In [11]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels = 2)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./models/distilbert",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=6,

    eval_strategy="epoch",
    save_strategy="epoch",

    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [18]:
from transformers import Trainer
from transformers import EarlyStoppingCallback
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, preds),
        "f1": f1_score(labels, preds)
    }

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train,
    eval_dataset=test,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

In [ ]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.013380,0.013783,0.998529,0.998483
2,0.005223,0.011133,0.998416,0.998366
3,0.001060,0.013721,0.998642,0.998599
4,0.000002,0.011649,0.998982,0.998949
5,0.000000,0.011697,0.998982,0.998949


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
trainer.evaluate()

{'eval_loss': 0.012364929541945457,
 'eval_accuracy': 0.9990948178320888,
 'eval_f1': 0.9990660751809479,
 'eval_runtime': 73.4153,
 'eval_samples_per_second': 120.384,
 'eval_steps_per_second': 15.051,
 'epoch': 6.0}

In [21]:
trainer.save_model("./models/distilbert_model")
tokenizer.save_pretrained("./models/distilbert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./models/distilbert_model/tokenizer_config.json',
 './models/distilbert_model/tokenizer.json')

In [3]:
from src.predict_distilbert import predict_news_distilbert

predict_news_distilbert("Government confirms secret alien research facility discovered in desert")

Loading weights: 100%|██████████| 104/104 [00:00<00:00, 16380.31it/s]


('FAKE', 1.0)